# 05 — Deploy App

Uploads the app source to Workspace Files and deploys it as a Databricks App using the SDK.

**Before running:** the `app/` folder from this repo must be uploaded to the same workspace location as these notebooks.

Re-run safe — every step is idempotent.

In [ ]:
dbutils.widgets.text("app_name",       "mt-genie-app",                   "App Name (lowercase, hyphens only)")
dbutils.widgets.text("warehouse_id",   "",                               "SQL Warehouse ID")
dbutils.widgets.text("genie_space_id", "01f131fb36aa184b8b53f281573aeb01", "Genie Space ID")
dbutils.widgets.text("catalog",        "medtech",                         "Catalog")
dbutils.widgets.text("schema",         "sales",                           "Schema")

import re

app_name       = dbutils.widgets.get("app_name").strip()
warehouse_id   = dbutils.widgets.get("warehouse_id").strip()
genie_space_id = dbutils.widgets.get("genie_space_id").strip()
catalog        = dbutils.widgets.get("catalog").strip()
schema         = dbutils.widgets.get("schema").strip()

if not warehouse_id:
    raise ValueError("warehouse_id is required")
if not genie_space_id:
    raise ValueError("genie_space_id is required — run notebook 04 first")
if not re.match(r'^[a-z][a-z0-9-]*$', app_name):
    raise ValueError("app_name must be lowercase letters, numbers, and hyphens only")

print(f"App name:    {app_name}")
print(f"Warehouse:   {warehouse_id}")
print(f"Genie space: {genie_space_id}")
print(f"Catalog:     {catalog}.{schema}")

## Step 1 — Upload App Source to Workspace

In [ ]:
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

w = WorkspaceClient()

# Resolve the app/ folder relative to this notebook's location
notebook_dir = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
app_src_ws  = f"{notebook_dir}/../app"   # workspace path to app/ folder
app_dest_ws = f"/Workspace/Shared/genie-workshop/{app_name}"

print(f"Source:      /Workspace{app_src_ws}")
print(f"Destination: {app_dest_ws}")

# Create destination directory
w.workspace.mkdirs(path=app_dest_ws)

for fname in ["app.py", "app.yaml", "index.html", "requirements.txt"]:
    src_path  = f"/Workspace{app_src_ws}/{fname}"
    dest_path = f"{app_dest_ws}/{fname}"

    with open(src_path, "rb") as f:
        content = f.read()

    # Substitute warehouse and genie space IDs in app.yaml
    if fname == "app.yaml":
        content = (
            content
            .replace(b"__WAREHOUSE_ID__",   warehouse_id.encode())
            .replace(b"__GENIE_SPACE_ID__", genie_space_id.encode())
        )

    import base64, requests
    cfg     = w.config
    host    = cfg.host.rstrip("/")
    headers = {**dict(cfg.authenticate()), "Content-Type": "application/json"}

    resp = requests.post(
        f"{host}/api/2.0/workspace/import",
        headers=headers,
        json={
            "path":      dest_path,
            "format":    "AUTO",
            "content":   base64.b64encode(content).decode(),
            "overwrite": True,
        },
    )
    status = "OK" if resp.status_code == 200 else f"ERROR ({resp.status_code}) {resp.text[:100]}"
    print(f"  {fname:<20} {status}")

print(f"\nApp source at: {app_dest_ws}")

## Step 2 — Create or Deploy App

In [ ]:
import time

# Create app if it doesn't exist
try:
    app = w.apps.get(name=app_name)
    print(f"App '{app_name}' exists — redeploying.")
except Exception:
    print(f"Creating app '{app_name}' ...")
    w.apps.create(
        name=app_name,
        description="J&J MedTech Sales Genie Workshop App",
    )
    print("App created. Waiting for compute to initialize...")
    time.sleep(10)

# Deploy from workspace path
print(f"Deploying from {app_dest_ws} ...")
deployment = w.apps.deploy(
    app_name=app_name,
    source_code_path=app_dest_ws,
    mode="SNAPSHOT",
).result()  # blocks until complete

print(f"Deployment state: {deployment.status.state}")
if str(deployment.status.state) == "DeploymentState.SUCCEEDED":
    print("Deployment succeeded.")
else:
    raise Exception(f"Deployment failed: {deployment.status.message}")

## Step 3 — Grant Permissions

In [ ]:
app_info     = w.apps.get(name=app_name)
sp_client_id = app_info.service_principal_client_id
cfg          = w.config
host         = cfg.host.rstrip("/")
headers      = {**dict(cfg.authenticate()), "Content-Type": "application/json"}

if sp_client_id:
    print(f"App SP client_id: {sp_client_id}")

    # Genie space CAN_MANAGE
    resp = requests.patch(
        f"{host}/api/2.0/permissions/genie/{genie_space_id}",
        headers=headers,
        json={"access_control_list": [{"service_principal_name": sp_client_id, "permission_level": "CAN_MANAGE"}]},
    )
    print(f"  Genie CAN_MANAGE:  {'OK' if resp.status_code == 200 else f'ERROR ({resp.status_code})'}")

    # Warehouse CAN_USE
    resp = requests.patch(
        f"{host}/api/2.0/permissions/warehouses/{warehouse_id}",
        headers=headers,
        json={"access_control_list": [{"service_principal_name": sp_client_id, "permission_level": "CAN_USE"}]},
    )
    print(f"  Warehouse CAN_USE: {'OK' if resp.status_code == 200 else f'ERROR ({resp.status_code})'}")

    # Unity Catalog grants
    for sql in [
        f"GRANT USE_CATALOG ON CATALOG {catalog} TO `{sp_client_id}`",
        f"GRANT USE_SCHEMA   ON SCHEMA  {catalog}.{schema} TO `{sp_client_id}`",
        f"GRANT SELECT       ON SCHEMA  {catalog}.{schema} TO `{sp_client_id}`",
    ]:
        try:
            spark.sql(sql)
            print(f"  OK  {sql}")
        except Exception as e:
            print(f"  ~   {sql}  ({e})")
else:
    print("WARNING: App SP not found. Re-run this cell after the app reaches RUNNING state.")

## Step 4 — Start App and Print URL

In [ ]:
# Start app
requests.post(f"{host}/api/2.0/apps/{app_name}/start", headers=headers)
print("App start requested. Waiting for RUNNING state ...")

app_url = None
for i in range(30):
    time.sleep(10)
    data    = requests.get(f"{host}/api/2.0/apps/{app_name}", headers=headers).json()
    state   = (data.get("app_status") or data.get("status") or {}).get("state", "")
    message = (data.get("app_status") or data.get("status") or {}).get("message", "")
    app_url = data.get("url", "")
    print(f"  [{i+1:02d}/30] {state:<12} {message[:60]}")
    if state == "RUNNING":
        break
    if state in ("CRASHED", "STOPPED"):
        print(f"App {state}. Check logs: Apps > {app_name} > Logs")
        break

if app_url:
    displayHTML(f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
                padding:28px 32px;background:#fff;border:1px solid #e0e0e0;
                border-radius:12px;max-width:600px;margin:16px 0">
      <div style="color:#EB1700;font-size:22px;font-weight:700;margin-bottom:8px">Setup Complete</div>
      <div style="margin-bottom:16px">
        <span style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:.5px">App URL</span><br>
        <a href="{app_url}" target="_blank" style="font-size:16px;color:#1565C0">{app_url}</a>
      </div>
      <div>
        <span style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:.5px">Genie Space ID</span><br>
        <code style="font-size:14px;color:#333">{genie_space_id}</code>
      </div>
    </div>
    """)
else:
    print(f"App URL not available yet. Check: {host}/apps/{app_name}")